### Boundary visualization: PLE 50Hz vs FixedPattern 50Hz (YouTube 10s)

This notebook compares **PLE 50Hz** vs **FixedPattern 50Hz** using the YouTube clips already prepared in:

- `/home/hoyso/projects/AudioTokenization/eval/youtube_10s/wav_16k/`

For each 10s clip, it:

- finds the **loudest 1-second window** (max RMS energy)
- runs the codec **encoder level-1** and the **DTP module** to get the boolean `mask`
- draws **red dashed vertical lines** at positions where **`mask == True`**
- annotates the **token indices** (small red numbers at the top)

Notes:
- The `mask` comes from `DTMAE/dtp/ops.py` and is a *frontier/kept-token* mask.
- Token time mapping uses `cfg.model.codec_encoder.hop_length` (samples) and `cfg.dataset.sample_rate`.
- Use a kernel/environment where `torch`, `torchaudio`, `omegaconf` work (e.g., `speech_eval`).



In [ ]:
from __future__ import annotations

import json
import math
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
from omegaconf import OmegaConf

REPO_ROOT = Path("/home/hoyso/projects/AudioTokenization").resolve()
EVAL_ROOT = REPO_ROOT / "eval"
DTMAE_ROOT = REPO_ROOT / "DTMAE"

# Ensure local imports work (mirrors eval/eval.py path setup, but minimal).
for p in (EVAL_ROOT, REPO_ROOT, DTMAE_ROOT):
    ps = str(p)
    if ps not in sys.path:
        sys.path.insert(0, ps)

from DTMAE.lightning_module import CodecLightningModule  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# DTMAE encoder uses FlashAttention kernels; CPU execution will fail.
if DEVICE != "cuda":
    raise RuntimeError(
        "CUDA is required for this notebook (FlashAttention kernels are CUDA-only). "
        "Please run with a GPU-enabled environment / CUDA-capable kernel."
    )

CLIP_MANIFEST = REPO_ROOT / "eval" / "youtube_10s" / "manifest.jsonl"
CLIP_WAV_DIR = REPO_ROOT / "eval" / "youtube_10s" / "wav_16k"
assert CLIP_MANIFEST.is_file(), f"Missing: {CLIP_MANIFEST}"
assert CLIP_WAV_DIR.is_dir(), f"Missing: {CLIP_WAV_DIR}"

RUN_PLE = REPO_ROOT / "results" / "results0117" / "default_PLE_50hz_vq16384"
RUN_FIXED = REPO_ROOT / "results" / "results0117" / "default_fixedpattern_50hz_vq65536"

PLE_EVAL_OUT = RUN_PLE / "eval_youtube_10s"
FIXED_EVAL_OUT = RUN_FIXED / "eval_youtube_10s"

DTP_STATS_PLE = RUN_PLE / "dtp_stats_ft" / "summary.json"  # best.fixed_tau
assert RUN_PLE.is_dir() and RUN_FIXED.is_dir()
assert (RUN_PLE / "hydra" / "config.yaml").is_file()
assert (RUN_FIXED / "hydra" / "config.yaml").is_file()
assert (RUN_PLE / "pl_log" / "last.ckpt").is_file()
assert (RUN_FIXED / "pl_log" / "last.ckpt").is_file()
assert DTP_STATS_PLE.is_file(), f"Missing: {DTP_STATS_PLE}"

print("OK: paths validated")



In [ ]:
def read_json(path: Path) -> dict:
    with path.open("r") as f:
        return json.load(f)


def read_jsonl(path: Path) -> List[dict]:
    with path.open("r") as f:
        return [json.loads(line) for line in f if line.strip()]


def resolve_repo_path(path_str: str) -> Path:
    p = Path(path_str)
    if p.is_absolute():
        return p
    return (REPO_ROOT / p).resolve()


@dataclass(frozen=True)
class Clip:
    filename: str
    wav_path: Path
    url: str
    start: str


clips: List[Clip] = []
for r in read_jsonl(CLIP_MANIFEST):
    wav_path = resolve_repo_path(str(r["clip_wav_16k"]))
    if not wav_path.is_file():
        # Fallback: look up by filename in CLIP_WAV_DIR
        wav_path = (CLIP_WAV_DIR / Path(str(r["clip_wav_16k"])).name).resolve()
    clips.append(
        Clip(
            filename=wav_path.name,
            wav_path=wav_path,
            url=str(r.get("url", "")),
            start=str(r.get("start", "")),
        )
    )

clips.sort(key=lambda c: c.filename)
print("clips:", len(clips))
print("example:", clips[0])



In [ ]:
def patch_legacy_dtp_state_dict(state_dict: Dict[str, torch.Tensor]) -> None:
    # Matches eval/eval.py
    legacy_keys = ["dtp.log_tau", "dtp.r_ema", "dtp.steps"]
    if not all(k in state_dict for k in legacy_keys):
        return

    log_tau = state_dict.pop("dtp.log_tau")
    tau = torch.exp(log_tau)
    state_dict["dtp.tau_train"] = tau.clone()
    state_dict["dtp.tau_eval"] = tau.clone()

    r_ema = state_dict.pop("dtp.r_ema")
    state_dict["dtp.r_ema_train"] = r_ema.clone()
    state_dict["dtp.r_ema_eval"] = r_ema.clone()

    steps = state_dict.pop("dtp.steps")
    state_dict["dtp.steps_train"] = steps.clone()
    state_dict["dtp.steps_eval"] = steps.clone()


def load_codec(run_dir: Path, fixed_tau: Optional[float] = None) -> Tuple[CodecLightningModule, object]:
    cfg_path = run_dir / "hydra" / "config.yaml"
    ckpt_path = run_dir / "pl_log" / "last.ckpt"
    assert cfg_path.is_file(), f"Missing: {cfg_path}"
    assert ckpt_path.is_file(), f"Missing: {ckpt_path}"

    cfg = OmegaConf.load(str(cfg_path))
    if fixed_tau is not None:
        # Ensure we replicate eval-time behavior: fixed tau overrides the Robbins–Monro controller.
        cfg = OmegaConf.merge(cfg, OmegaConf.create({"model": {"resampler": {"dtp_params": {"fixed_tau": float(fixed_tau)}}}}))

    model = CodecLightningModule(cfg=cfg).to(DEVICE).eval()

    state = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    state_dict = state.get("state_dict", state)
    patch_legacy_dtp_state_dict(state_dict)

    # Strict load; if this fails, it means the checkpoint/config mismatch.
    incompatible = model.load_state_dict(state_dict, strict=True)
    if getattr(incompatible, "missing_keys", None) or getattr(incompatible, "unexpected_keys", None):
        raise RuntimeError(f"State dict mismatch. missing={len(incompatible.missing_keys)}, unexpected={len(incompatible.unexpected_keys)}")

    return model, cfg


ple_tau = float(read_json(DTP_STATS_PLE)["best"]["fixed_tau"])
print("PLE best.fixed_tau:", ple_tau)

ple_model, ple_cfg = load_codec(RUN_PLE, fixed_tau=ple_tau)
fixed_model, fixed_cfg = load_codec(RUN_FIXED, fixed_tau=None)

print("Loaded models.")
print("PLE dtp_cls:", ple_cfg.model.resampler.dtp_cls)
print("Fixed dtp_cls:", fixed_cfg.model.resampler.dtp_cls)



In [ ]:
def load_wav_mono_16k(path: Path) -> torch.Tensor:
    wav, sr = torchaudio.load(str(path))
    if wav.dim() == 2 and wav.size(0) > 1:
        wav = wav[:1, :]
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    return wav


def loudest_1s_segment(wav_mono_16k: torch.Tensor, sr: int = 16000, dur_s: float = 1.0, step_s: float = 0.01) -> Tuple[torch.Tensor, int]:
    assert wav_mono_16k.dim() == 2 and wav_mono_16k.size(0) == 1
    x = wav_mono_16k[0]
    win = int(round(sr * dur_s))
    step = max(1, int(round(sr * step_s)))
    if x.numel() <= win:
        return wav_mono_16k[:, :win], 0

    # Compute window energy on a stride (fast, deterministic)
    sq = x.float().pow(2)
    idx = torch.arange(0, x.numel() - win + 1, step, device=x.device)
    energies = []
    for s in idx.tolist():
        energies.append(float(sq[s : s + win].mean().item()))

    best_i = int(np.argmax(np.asarray(energies)))
    best_start = int(idx[best_i].item())
    seg = x[best_start : best_start + win].unsqueeze(0)
    return seg, best_start


def stft_db(wav_mono_16k: torch.Tensor, n_fft: int = 1024, hop_length: int = 256) -> np.ndarray:
    # returns shape [freq, time]
    spec = torchaudio.transforms.Spectrogram(n_fft=n_fft, hop_length=hop_length, power=2.0)(wav_mono_16k)
    spec_db = torchaudio.transforms.AmplitudeToDB(stype="power")(spec)
    return spec_db.squeeze(0).cpu().numpy()


def get_mask_from_model(model: CodecLightningModule, cfg, wav_1s_mono_16k: torch.Tensor) -> Tuple[np.ndarray, float, float, int]:
    # wav_1s_mono_16k: [1, T]
    assert wav_1s_mono_16k.dim() == 2 and wav_1s_mono_16k.size(0) == 1
    wav = wav_1s_mono_16k.to(DEVICE)

    with torch.inference_mode():
        vq_emb = model.encoder(wav.unsqueeze(1), level=1)
        dtp_out = model.dtp(vq_emb)

    # Handle (mask, avg_r, tau) or (mask, avg_r, tau, aux_loss)
    if isinstance(dtp_out, (list, tuple)) and len(dtp_out) >= 3:
        mask_t = dtp_out[0]
        avg_r_t = dtp_out[1]
        tau_t = dtp_out[2]
    else:
        raise RuntimeError(f"Unexpected dtp output: {type(dtp_out)}")

    mask = mask_t.detach().bool().squeeze(0).cpu().numpy()  # [N]
    avg_r = float(avg_r_t.detach().cpu().item()) if torch.is_tensor(avg_r_t) else float(avg_r_t)
    tau_used = float(tau_t.detach().cpu().item()) if torch.is_tensor(tau_t) else float(tau_t)
    N = int(mask.shape[0])
    return mask, avg_r, tau_used, N


def plot_boundaries_on_spec(
    ax,
    spec_db: np.ndarray,
    sr: int,
    dur_s: float,
    boundary_idx: np.ndarray,
    token_hop_s: float,
    title: str,
    annotate_max: int = 120,
):
    # spec_db: [freq, time]
    extent = [0.0, dur_s, 0.0, sr / 2 / 1000.0]  # kHz
    im = ax.imshow(spec_db, origin="lower", aspect="auto", cmap="magma", extent=extent)
    ax.set_title(title)
    ax.set_xlabel("Time / s")
    ax.set_ylabel("Frequency / kHz")

    # Convert token indices to time positions
    times = boundary_idx.astype(np.float64) * float(token_hop_s)
    times = times[(times >= 0.0) & (times <= dur_s)]

    for t in times:
        ax.axvline(x=float(t), color="red", linestyle="--", linewidth=0.8, alpha=0.75)

    # Annotate token indices at top (may be dense; auto-throttle)
    if boundary_idx.size > 0:
        step = 1
        if boundary_idx.size > annotate_max:
            step = int(math.ceil(boundary_idx.size / annotate_max))

        for i in boundary_idx[::step]:
            t = float(i) * float(token_hop_s)
            if 0.0 <= t <= dur_s:
                ax.text(
                    t,
                    1.02,
                    str(int(i)),
                    transform=ax.get_xaxis_transform(),
                    rotation=90,
                    ha="center",
                    va="bottom",
                    fontsize=6,
                    color="red",
                    alpha=0.9,
                )

    return im



In [ ]:
SR = 16000
DUR_S = 1.0

# Token hop from cfg (samples -> seconds)
PLE_TOKEN_HOP_S = float(ple_cfg.model.codec_encoder.hop_length) / float(ple_cfg.dataset.sample_rate)
FIXED_TOKEN_HOP_S = float(fixed_cfg.model.codec_encoder.hop_length) / float(fixed_cfg.dataset.sample_rate)

print("PLE token hop (s):", PLE_TOKEN_HOP_S)
print("Fixed token hop (s):", FIXED_TOKEN_HOP_S)

# Choose which clips to plot
CLIP_INDICES = list(range(len(clips)))  # e.g., [0] for just the first

for ci in CLIP_INDICES:
    clip = clips[ci]
    wav10 = load_wav_mono_16k(clip.wav_path)
    seg1, start_sample = loudest_1s_segment(wav10, sr=SR, dur_s=DUR_S, step_s=0.01)
    start_s = start_sample / SR

    ple_mask, ple_avg_r, ple_tau_used, ple_N = get_mask_from_model(ple_model, ple_cfg, seg1)
    fixed_mask, fixed_avg_r, fixed_tau_used, fixed_N = get_mask_from_model(fixed_model, fixed_cfg, seg1)

    ple_idx = np.where(ple_mask)[0]
    fixed_idx = np.where(fixed_mask)[0]

    spec = stft_db(seg1, n_fft=1024, hop_length=256)

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2), constrained_layout=True)

    im0 = plot_boundaries_on_spec(
        axes[0],
        spec_db=spec,
        sr=SR,
        dur_s=DUR_S,
        boundary_idx=ple_idx,
        token_hop_s=PLE_TOKEN_HOP_S,
        title=f"PLE (hop={ple_cfg.model.codec_encoder.hop_length} samples, tau={ple_tau_used:.4f}, avg_r={ple_avg_r:.3f})",
    )

    im1 = plot_boundaries_on_spec(
        axes[1],
        spec_db=spec,
        sr=SR,
        dur_s=DUR_S,
        boundary_idx=fixed_idx,
        token_hop_s=FIXED_TOKEN_HOP_S,
        title=f"FixedPattern (stride~{fixed_tau_used:.1f}, avg_r={fixed_avg_r:.3f})",
    )

    fig.suptitle(f"{clip.filename} | loudest 1s @ {start_s:.2f}s (within 10s clip)\nurl={clip.url}")
    cbar = fig.colorbar(im1, ax=axes, fraction=0.025, pad=0.02)
    cbar.set_label("dB")
    plt.show()

    print(
        f"{clip.filename}: start_s={start_s:.2f} | PLE kept={ple_idx.size}/{ple_N} | Fixed kept={fixed_idx.size}/{fixed_N}"
    )

